# Logistic Regression (TensorFlow v2) + FGM Pipeline

This notebook mirrors the **NN + FGM + dual-stream consistency gate** workflow, but replaces the neural network with a **TensorFlow v2 logistic regression classifier**.

Pipeline steps:

1. Train a **clean logistic regression** model on the clean training split  
2. Evaluate the clean model on clean test data  
3. Generate **FGM adversarial** train and test samples  
4. Retrain a second logistic regression model using ART's **AdversarialTrainer**  
5. Evaluate both models on:
   - clean test data
   - adversarial test data
   - combined clean + adversarial test data
6. Apply the same **dual-stream consistency gate** used in the CNN / LSTM / Transformer / NN notebooks

Logistic regression here is implemented as a **single dense softmax layer** in TensorFlow v2.


In [37]:
# If needed, install once:
# !pip install tensorflow adversarial-robustness-toolbox scikit-learn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


In [38]:
# -----------------------------
# Configuration
# -----------------------------
DATA_PATH = Path(r"..\CSVs\dataset.csv") 

LABEL_COL = "anomaly"



# Drop non-feature columns if needed
DROP_COLS = {LABEL_COL, "segment", "train", "sampling", "channel"}

TEST_SIZE = 0.2

# Logistic regression hyperparameters
LR = 1e-3
L2_REG = 1e-4
BATCH_SIZE = 128
NB_EPOCHS = 20

# FGM settings
FGM_EPS = 0.10

# Adversarial training settings
ADV_RATIO = 0.55

# Save paths
SAVE_MODELS = True
ARTIFACT_DIR = Path(r"Artifacts\LogisticRegression")
RESULTS_DIR = Path(r"Results\LogisticRegressionResults")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [39]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()
    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )
    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), scaler, feature_cols


def to_one_hot(y: np.ndarray, num_classes: int = 2):
    return tf.keras.utils.to_categorical(y, num_classes=num_classes).astype(np.float32)


if not DATA_PATH.exists():
    raise ValueError(
        f"Set DATA_PATH first. Current value does not exist: {DATA_PATH}"
    )

X_train, X_test, y_train, y_test, scaler, feature_cols = load_and_prepare(str(DATA_PATH))

y_train_oh = to_one_hot(y_train, 2)
y_test_oh = to_one_hot(y_test, 2)


Loaded: ..\CSVs\dataset.csv
Rows=2123, Features=18, Label dist=[1689  434]
Train=(1698, 18), Test=(425, 18)


In [40]:
def build_logistic_model(d_in: int, l2_reg: float = L2_REG):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(d_in,)),
        tf.keras.layers.Dense(
            2,
            activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
            name="logistic_output",
        ),
    ])


def make_art_classifier(d_in: int, lr: float = LR, l2_reg: float = L2_REG):
    model = build_logistic_model(d_in=d_in, l2_reg=l2_reg)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    loss_object = tf.keras.losses.CategoricalCrossentropy()

    @tf.function
    def train_step(model_instance, x_batch, y_batch):
        with tf.GradientTape() as tape:
            predictions = model_instance(x_batch, training=True)
            loss = loss_object(y_batch, predictions)
            if model_instance.losses:
                loss += tf.add_n(model_instance.losses)

        gradients = tape.gradient(loss, model_instance.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model_instance.trainable_variables))

    classifier = TensorFlowV2Classifier(
        model=model,
        nb_classes=2,
        input_shape=(d_in,),
        loss_object=loss_object,
        train_step=train_step,
    )
    return classifier


def predict_labels(art_clf, X: np.ndarray):
    probs = art_clf.predict(X).astype(np.float32)
    return np.argmax(probs, axis=1)


def eval_classifier(art_clf, X: np.ndarray, y_true: np.ndarray, name: str):
    y_pred = predict_labels(art_clf, X)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    metrics = {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }
    return metrics, y_pred


def save_art_model_state(art_clf, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    art_clf.model.save_weights(str(out_path))
    print(f"Saved model weights: {out_path}")


In [41]:
# Step 1: Train clean logistic regression model
print("Training clean TensorFlow v2 logistic regression with:", {
    "learning_rate": LR,
    "l2_reg": L2_REG,
    "batch_size": BATCH_SIZE,
    "epochs": NB_EPOCHS,
    "fgm_eps": FGM_EPS,
    "adv_ratio": ADV_RATIO,
})

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train_oh, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

clean_on_clean, y_pred_clean = eval_classifier(
    art_clean,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

if SAVE_MODELS:
    save_art_model_state(art_clean, ARTIFACT_DIR / "logreg_tf2_clean.weights.h5")


Training clean TensorFlow v2 logistic regression with: {'learning_rate': 0.001, 'l2_reg': 0.0001, 'batch_size': 128, 'epochs': 20, 'fgm_eps': 0.1, 'adv_ratio': 0.55}

[clean_model_on_clean_test] acc=0.7953 f1=0.0000
confusion matrix:
[[338   0]
 [ 87   0]]
              precision    recall  f1-score   support

           0     0.7953    1.0000    0.8860       338
           1     0.0000    0.0000    0.0000        87

    accuracy                         0.7953       425
   macro avg     0.3976    0.5000    0.4430       425
weighted avg     0.6325    0.7953    0.7046       425

Saved model weights: Artifacts\LogisticRegression\logreg_tf2_clean.weights.h5


In [42]:
# Step 2: Initialize FGM and generate adversarial samples
fgm = FastGradientMethod(estimator=art_clean, eps=FGM_EPS)

X_train_adv = fgm.generate(x=X_train)
X_test_adv = fgm.generate(x=X_test)

print("Adversarial data generated:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# Optional predicted labels on adversarial inputs
y_train_adv_pred = predict_labels(art_clean, X_train_adv)
y_test_adv_pred = predict_labels(art_clean, X_test_adv)

# Ground-truth labels stay aligned with the original data
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()
y_train_adv_oh = to_one_hot(y_train_adv, 2)
y_test_adv_oh = to_one_hot(y_test_adv, 2)

# Combined clean + adversarial test set
X_test_combined = np.concatenate([X_test, X_test_adv], axis=0).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv], axis=0).astype(np.int64)


Adversarial data generated:
X_train_adv: (1698, 18)
X_test_adv: (425, 18)


In [43]:
# Evaluate clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_classifier(
    art_clean,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_classifier(
    art_clean,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)



[clean_model_on_adv_test] acc=0.5435 f1=0.3782
confusion matrix:
[[172 166]
 [ 28  59]]
              precision    recall  f1-score   support

           0     0.8600    0.5089    0.6394       338
           1     0.2622    0.6782    0.3782        87

    accuracy                         0.5435       425
   macro avg     0.5611    0.5935    0.5088       425
weighted avg     0.7376    0.5435    0.5859       425


[clean_model_on_combined_test] acc=0.6694 f1=0.2957
confusion matrix:
[[510 166]
 [115  59]]
              precision    recall  f1-score   support

           0     0.8160    0.7544    0.7840       676
           1     0.2622    0.3391    0.2957       174

    accuracy                         0.6694       850
   macro avg     0.5391    0.5468    0.5399       850
weighted avg     0.7026    0.6694    0.6841       850



In [44]:
# Step 3: Adversarial training with ART's AdversarialTrainer
art_adv = make_art_classifier(d_in=X_train.shape[1])

adv_trainer = AdversarialTrainer(
    classifier=art_adv,
    attacks=fgm,
    ratio=ADV_RATIO,
)

adv_trainer.fit(
    X_train,
    y_train_oh,
    batch_size=BATCH_SIZE,
    nb_epochs=NB_EPOCHS,
)

if SAVE_MODELS:
    save_art_model_state(art_adv, ARTIFACT_DIR / "logreg_tf2_adversarial_trained.weights.h5")


Adversarial training epochs: 100%|██████████| 20/20 [00:04<00:00,  4.76it/s]

Saved model weights: Artifacts\LogisticRegression\logreg_tf2_adversarial_trained.weights.h5


In [45]:
# Step 4: Evaluate adversarially trained model
adv_trained_on_adv, y_pred_adv = eval_classifier(
    art_adv,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
    art_adv,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
    art_adv,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)



[adv_trained_model_on_adv_test] acc=0.7953 f1=0.0000
confusion matrix:
[[338   0]
 [ 87   0]]
              precision    recall  f1-score   support

           0     0.7953    1.0000    0.8860       338
           1     0.0000    0.0000    0.0000        87

    accuracy                         0.7953       425
   macro avg     0.3976    0.5000    0.4430       425
weighted avg     0.6325    0.7953    0.7046       425


[adv_trained_model_on_clean_test] acc=0.7953 f1=0.0000
confusion matrix:
[[338   0]
 [ 87   0]]
              precision    recall  f1-score   support

           0     0.7953    1.0000    0.8860       338
           1     0.0000    0.0000    0.0000        87

    accuracy                         0.7953       425
   macro avg     0.3976    0.5000    0.4430       425
weighted avg     0.6325    0.7953    0.7046       425


[adv_trained_model_on_combined_test] acc=0.7953 f1=0.0000
confusion matrix:
[[676   0]
 [174   0]]
              precision    recall  f1-score   support


In [46]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_clean,
    adv_trained_on_adv,
    adv_trained_on_combined,
])

display(summary_df)

summary_path = RESULTS_DIR / "logreg_tf2_fgm_pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")


,model_eval,acc,f1
0,clean_model_on_clean_test,0.795294,0.000000
1,clean_model_on_adv_test,0.543529,0.378205
2,clean_model_on_combined_test,0.669412,0.295739
3,adv_trained_model_on_clean_test,0.795294,0.000000
4,adv_trained_model_on_adv_test,0.795294,0.000000
5,adv_trained_model_on_combined_test,0.795294,0.000000


Saved: Results\LogisticRegressionResults\logreg_tf2_fgm_pipeline_summary.csv


## Dual-Stream Consistency Gate

This section uses the same dual-stream detector structure as the CNN, LSTM, Transformer, and updated NN notebooks.

### Gate logic
- **Nominal model** = clean logistic regression model  
- **Guardian model** = adversarially trained logistic regression model  
- Flag likely attacks using:
  1. prediction disagreement
  2. high-confidence disagreement
  3. nominal predicts benign while guardian predicts anomaly by a large enough margin

The gate is evaluated both as:
- a **final prediction system** for clean / adversarial / combined inputs
- an **attack detector** using **FPR**, **TPR**, and **F1**


In [47]:
CONFIDENCE_THRESHOLD = 0.20
DISAGREEMENT_THRESHOLD = 0.55

def predict_with_confidence(art_clf, X: np.ndarray):
    probs = art_clf.predict(X).astype(np.float32)
    preds = np.argmax(probs, axis=1)
    return preds, probs


class DualStreamDetector:
    def __init__(self, nominal_model, guardian_model):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal = predict_with_confidence(self.nominal_model, X)
        preds_guardian, probs_guardian = predict_with_confidence(self.guardian_model, X)

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        details = []

        for i in range(n_samples):
            yA = int(preds_nominal[i])
            yB = int(preds_guardian[i])
            pA = probs_nominal[i]
            pB = probs_guardian[i]

            conf_nominal = float(pA[yA])
            conf_guardian = float(pB[yB])

            detected = False
            reasons = []

            if yA != yB:
                detected = True
                reasons.append("disagreement")

            if yA == 0 and yB == 1:
                prob_diff = float(pB[1] - pA[1])
                if conf_nominal >= confidence_threshold and conf_guardian >= confidence_threshold:
                    if prob_diff >= disagreement_threshold:
                        detected = True
                        reasons.append("high_confidence_disagreement")
            else:
                prob_diff = float(pB[1] - pA[1])

            flags[i] = int(detected)
            details.append({
                "nominal_pred": yA,
                "guardian_pred": yB,
                "nominal_conf": conf_nominal,
                "guardian_conf": conf_guardian,
                "nominal_anom_prob": float(pA[1]),
                "guardian_anom_prob": float(pB[1]),
                "prob_diff_anomaly": prob_diff,
                "detected": bool(detected),
                "reason": ",".join(reasons) if reasons else "none",
            })

        final_preds = np.where(flags == 1, preds_guardian, preds_nominal)
        details_df = pd.DataFrame(details)
        return final_preds, flags, details_df


def evaluate_dual_stream_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


detector = DualStreamDetector(
    nominal_model=art_clean,
    guardian_model=art_adv,
)

gated_clean_pred, flags_clean, gate_clean_df = detector.detect_attacks(X_test)
gated_adv_pred, flags_adv, gate_adv_df = detector.detect_attacks(X_test_adv)
gated_combined_pred, flags_combined, gate_combined_df = detector.detect_attacks(X_test_combined)

gated_clean_metrics = evaluate_dual_stream_predictions(
    y_test, gated_clean_pred, "dual_stream_final_predictions_on_clean_test"
)
gated_adv_metrics = evaluate_dual_stream_predictions(
    y_test_adv, gated_adv_pred, "dual_stream_final_predictions_on_adv_test"
)
gated_combined_metrics = evaluate_dual_stream_predictions(
    y_test_combined, gated_combined_pred, "dual_stream_final_predictions_on_combined_test"
)

dual_stream_prediction_summary_df = pd.DataFrame([
    gated_clean_metrics,
    gated_adv_metrics,
    gated_combined_metrics,
])

print("\nDual-stream final prediction summary:")
display(dual_stream_prediction_summary_df)

y_attack_true = np.concatenate([
    np.zeros(len(flags_clean), dtype=int),
    np.ones(len(flags_adv), dtype=int),
])
y_attack_pred = np.concatenate([flags_clean, flags_adv])

dual_stream_detection_results = pd.DataFrame([{
    "attack": "FGM",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "disagreement_threshold": DISAGREEMENT_THRESHOLD,
    "FPR": float(flags_clean.mean()),
    "TPR": float(flags_adv.mean()),
    "F1": float(f1_score(y_attack_true, y_attack_pred, zero_division=0)),
    "clean_model_acc_on_adv": float(clean_on_adv["acc"]),
    "guardian_model_acc_on_clean": float(adv_trained_on_clean["acc"]),
    "guardian_model_acc_on_adv": float(adv_trained_on_adv["acc"]),
}])

print("\nDual-stream attack-detection summary:")
display(dual_stream_detection_results)

print("\nGate reason counts on clean test:")
display(gate_clean_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))

print("\nGate reason counts on adversarial test:")
display(gate_adv_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))



[dual_stream_final_predictions_on_clean_test] acc=0.7953 f1=0.0000
confusion matrix:
[[338   0]
 [ 87   0]]
              precision    recall  f1-score   support

           0     0.7953    1.0000    0.8860       338
           1     0.0000    0.0000    0.0000        87

    accuracy                         0.7953       425
   macro avg     0.3976    0.5000    0.4430       425
weighted avg     0.6325    0.7953    0.7046       425


[dual_stream_final_predictions_on_adv_test] acc=0.7953 f1=0.0000
confusion matrix:
[[338   0]
 [ 87   0]]
              precision    recall  f1-score   support

           0     0.7953    1.0000    0.8860       338
           1     0.0000    0.0000    0.0000        87

    accuracy                         0.7953       425
   macro avg     0.3976    0.5000    0.4430       425
weighted avg     0.6325    0.7953    0.7046       425


[dual_stream_final_predictions_on_combined_test] acc=0.7953 f1=0.0000
confusion matrix:
[[676   0]
 [174   0]]
              prec

,model_eval,acc,f1
0,dual_stream_final_predictions_on_clean_test,0.795294,0.0
1,dual_stream_final_predictions_on_adv_test,0.795294,0.0
2,dual_stream_final_predictions_on_combined_test,0.795294,0.0



Dual-stream attack-detection summary:


,attack,confidence_threshold,disagreement_threshold,FPR,TPR,F1,clean_model_acc_on_adv,guardian_model_acc_on_clean,guardian_model_acc_on_adv
0,FGM,0.2,0.55,0.0,0.529412,0.692308,0.543529,0.795294,0.795294



Gate reason counts on clean test:


,reason,count
0,none,425



Gate reason counts on adversarial test:


,reason,count
0,disagreement,225
1,none,200


In [48]:
# Save dual-stream outputs
dual_stream_prediction_summary_path = RESULTS_DIR / "logreg_tf2_dual_stream_prediction_summary.csv"
dual_stream_detection_path = RESULTS_DIR / "logreg_tf2_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(RESULTS_DIR / "logreg_tf2_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")


Saved: Results\LogisticRegressionResults\logreg_tf2_dual_stream_prediction_summary.csv
Saved: Results\LogisticRegressionResults\logreg_tf2_dual_stream_detection_results.csv
